In [0]:
%sql
create catalog cdc_demo;
use catalog cdc_demo

In [0]:
%sql
create schema bronze;
create schema silver;
create schema gold;


In [0]:
%sql
create schema landing;

In [0]:
%sql
create volume cdc_demo.landing.raw_files

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    max,
    min,
    round,
    sum,
)

In [0]:
df = (
    spark.read.format('csv')
    .option('header', 'true')
    .option('inferSchema', 'true')
    .load('/Volumes/cdc_demo/landing/raw_files/online_retail_II.csv')
)

In [0]:
df.limit(100).display()

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00.000Z,6.95,13085.0,United Kingdom
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom
489434,22041,"""RECORD FRAME 7"""" SINGLE SIZE """,48,2009-12-01T07:45:00.000Z,2.1,13085.0,United Kingdom
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom
489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01T07:45:00.000Z,1.65,13085.0,United Kingdom
489434,21871,SAVE THE PLANET MUG,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom
489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01T07:45:00.000Z,5.95,13085.0,United Kingdom
489435,22350,CAT BOWL,12,2009-12-01T07:46:00.000Z,2.55,13085.0,United Kingdom
489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01T07:46:00.000Z,3.75,13085.0,United Kingdom


In [0]:
df.printSchema()

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: double (nullable = true)
 |-- Country: string (nullable = true)



In [0]:
df.count()

1067371

In [0]:
df.filter(col('Customer ID').isNull()).show()

+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer ID|       Country|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
| 489464|    21733|        85123a mixed|     -96|2009-12-01 10:52:00|  0.0|       NULL|United Kingdom|
| 489463|    71477|               short|    -240|2009-12-01 10:52:00|  0.0|       NULL|United Kingdom|
| 489467|   85123A|         21733 mixed|    -192|2009-12-01 10:53:00|  0.0|       NULL|United Kingdom|
| 489521|    21646|                NULL|     -50|2009-12-01 11:44:00|  0.0|       NULL|United Kingdom|
| 489525|   85226C|BLUE PULL BACK RA...|       1|2009-12-01 11:49:00| 0.55|       NULL|United Kingdom|
| 489525|    85227|SET/6 3D KIT CARD...|       1|2009-12-01 11:49:00| 0.85|       NULL|United Kingdom|
| 489548|    22271|FELTCRAFT DOLL ROSIE|       1|2009-12-01 12:32:00| 2.9

In [0]:
df.filter(col('Customer ID').isNull()).count()

824364

In [0]:
df_non_null = df.filter(col('Customer ID').isNotNull())

df_non_null.select('Customer ID').distinct().count()

5942

In [0]:
df.select('Customer ID').where(col('Customer ID').isNotNull()).distinct().count()

5942

In [0]:

df_non_null.select(
    countDistinct('Customer ID').alias('distinct_customer_count')
).display()

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-4797211912914475>, line 3
      1 from pyspark.sql.functions import *
----> 3 dfnonnull.select(
      4     countDistinct('Customer ID').alias("distinct_customer_count")
      5     ).display()

NameError: name 'dfnonnull' is not defined

In [0]:
(
    df_non_null.groupBy('Customer ID')
    .agg(count('*').alias('row_count'))
    .orderBy('Customer ID')
    .display()
)

Customer ID,Count
12346.0,48
12347.0,253
12348.0,51
12349.0,180
12350.0,17
12351.0,21
12352.0,113
12353.0,24
12354.0,58
12355.0,35


In [0]:

(
    df.filter(col('Customer ID').isNotNull())
    .groupBy('Customer ID')
    .agg(countDistinct('Country').alias('country_count'))
    .filter(col('country_count') > 1)
    .orderBy('Customer ID')
    .display()
)

Customer ID,countCountry
12370.0,2
12394.0,2
12413.0,2
12417.0,2
12422.0,2
12423.0,2
12429.0,2
12431.0,2
12449.0,2
12455.0,2


In [0]:
df.select(
    min('InvoiceDate').alias('min_invoice_date'),
    max('InvoiceDate').alias('max_invoice_date'),
).display()

min(InvoiceDate),max(InvoiceDate)
2009-12-01T07:45:00.000Z,2011-12-09T12:50:00.000Z


In [0]:
cancelled_invoices = df.filter(col('Invoice').startswith('C'))

cancelled_invoices.count()

19494

In [0]:
non_positive_rows = df.filter((col('Price') <= 0) | (col('Quantity') <= 0))

non_positive_rows.count()

25700

In [0]:
customer_spend = (
    df.filter(
        col('Customer ID').isNotNull()
        & (col('Price') > 0)
        & (col('Quantity') > 0)
    )
    .withColumn('spend', round(col('Quantity') * col('Price'), 3))
    .groupBy('Customer ID')
    .agg(sum('spend').alias('total_spend'))
)

customer_spend.display()

Customer ID,TotalSpend
13457.0,148.22
13416.0,940.8
14286.0,8479.76
17047.0,2289.23
14285.0,3284.42
16782.0,10438.410000000007
17866.0,1134.9499999999998
13050.0,12344.529999999999
15184.0,689.95
17776.0,338.56


In [0]:
customer_spend.selectExpr(
    'percentile_approx(total_spend, array(0.5, 0.8, 0.95)) AS percentiles'
).display()

Percentiles
"List(897.6200000000003, 2957.47, 9530.080000000002)"
